In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
#import dataset

df=pd.read_csv('Crop_recommendation.csv')
df.describe()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.drop('label',axis=1).corr(),annot=True,cmap='coolwarm')
plt.title("corr soil and weather")
plt.show()

In [ ]:
#split label dan fitur

X = df.drop('label',axis=1)

y = df['label']

In [ ]:
#scaling biar seimbang
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#encode crop jadi integer etc. apel = 0
label_encode = LabelEncoder()
y_encoded = label_encode.fit_transform(y)

crop_dict=dict(zip(label_encode.classes_,label_encode.transform(label_encode.classes_)))
print(crop_dict)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training data size: {X_train.shape}")
print(f"Testing data size: {X_test.shape}")

In [ ]:
# Initialize the model
# random state = seed random diatur agar data yang diacak tetap sama
# n_estimators = Jumlah pohon, tambah banyak = tambah training, diminished alert
# max_depth = panjang turunan cabang, maximal berapa kali split
# min_samples_split = minimal berapa cabang utk 1 kali split
# max_features = maximal label referensi per split

model = RandomForestClassifier(
    random_state=42,
    n_estimators=50,
    max_depth = 8,
    min_samples_split = 5,
    max_features = 'sqrt'

    )

# Train (fit) the model on the training data
model.fit(X_train, y_train)


In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Check the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# See a detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encode.classes_))


In [ ]:
# Import grid search
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# 1. Start with a blank model
rf_model = RandomForestClassifier(random_state=42)

# 2. Define the "Grid" of parameters you want the computer to test
# It will test every possible combination of these numbers!
param_grid = {
    'n_estimators': [100, 200, 300],       # Number of trees
    'max_depth': [None, 5, 10, 20],        # How deep the trees can go
    'min_samples_split': [2, 5, 10]        # Minimum samples to split
}

print("Starting Grid Search... this might take a minute...")

# 3. Setup the Grid Search
# cv=5 means it runs "5-Fold Cross Validation" to ensure the score isn't a fluke
grid_search = GridSearchCV(
    estimator=rf_model, 
    param_grid=param_grid, 
    cv=5,               
    n_jobs=-1,          # Uses all your CPU cores to make it faster!
    verbose=1           # Prints progress updates to your screen
)

# 4. Let it test all 36 combinations! (3 * 4 * 3 = 36)
grid_search.fit(X_train, y_train)

# 5. Print out the absolute best parameters it found
print("\n🔥 Grid Search Finished! 🔥")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_ * 100:.2f}%")

# 6. Automatically extract the winner as your final model!
best_model = grid_search.best_estimator_


In [ ]:
# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Check the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# See a detailed report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encode.classes_))
